# Matching investigation

Runs the **real, current matching pipeline** (`backend/app/services/nutrition_mapping.map_item`, `matcher.py`, `base_terms.py`, `fallback_categories.py`, `off_api.py` — nothing reimplemented) against every `receipt_items` row currently in the database.

For each item this shows:
- the **raw text extracted** from the receipt (`raw_name`) and the cleaned `normalized_name`
- the exact **text passed to the OpenFoodFacts search** (`off_query`) — this can differ from `normalized_name` when the matcher falls back to a generalised term (e.g. "Rispentomaten" -> "Tomate")
- the **match currently used** (name, OFF id, confidence, match type, nutrition)
- **how many alternatives** in that same OFF search response also carry usable nutrition data, and their **names** — i.e. candidates that existed but weren't chosen (or weren't even considered, in the fallback case)

Run cells top to bottom. Repeat runs are fast/cheap — `off_api.py` caches every OpenFoodFacts response on disk (`backend/app/services/_off_cache.json`), so only genuinely new queries hit the network.

In [1]:
import sys
from pathlib import Path

# Make sure `backend` is importable regardless of where Jupyter's cwd ends up.
_here = Path.cwd()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "backend").is_dir():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise RuntimeError("Could not find the repo root (a parent containing 'backend/') from cwd=" + str(_here))

import pandas as pd

from backend.app.db.supabase import get_all_receipt_items
from backend.app.services import off_api, matcher, base_terms, fallback_categories, nutrition_mapping
from backend.app.models.nutrition import MatchType

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 200)

## Load all receipt items

`get_all_receipt_items()` is the same dev/debug helper the backend itself uses for full-DB sanity checks (see `nutrition_snapshot.build_snapshot_from_all_receipts`) — every row in `receipt_items`, across every receipt and every session.

In [2]:
items = get_all_receipt_items()
print(f"{len(items)} receipt_items rows loaded")
items[0] if items else None

166 receipt_items rows loaded


{'id': '1c75ce3d-07cc-481d-a280-74a6da9e2e75',
 'receipt_id': '96d33a71-ea37-4d0a-a185-54cd2d1734a0',
 'raw_name': 'RUCOLA',
 'normalized_name': 'Rucola',
 'quantity': '1',
 'price': None,
 'confidence': 0.9,
 'unit': '',
 'category': 'Gemüse',
 'matched_product_id': None}

## The investigation function

This mirrors `nutrition_mapping.map_item()` tier-for-tier, calling the exact same production functions (`matcher.match_product`, `base_terms.generic_term`, `fallback_categories.fallback_nutrition`) so the "current match" reported here is identical to what the running app would produce. The only addition is capturing the **full OFF candidate pool** for whichever query tier actually decided the outcome, so we can report how many of those candidates carry real nutrition data and what they're called.

- **Tier 1** — search OFF with the item's own (normalized) name.
- **Tier 2** — if tier 1 found nothing usable, reduce to a generic produce term (e.g. "Rispentomaten" -> "Tomate") and retry.
- **Tier 3** — if neither worked, fall back to a category-based nutrition estimate (no OFF match at all).

In [3]:
def investigate_item(item: dict) -> dict:
    """Run the real matching pipeline for one receipt_items row and report
    the chosen match plus the alternatives it was chosen over."""

    raw_name = item.get("raw_name")
    normalized_name = item.get("normalized_name")
    category = item.get("category")
    # Same name-resolution the production pipeline uses (prefers normalized_name).
    query_name = nutrition_mapping._item_name(item)

    result = {
        "receipt_item_id": item.get("id"),
        "receipt_id": item.get("receipt_id"),
        "raw_name": raw_name,
        "normalized_name": normalized_name,
        "off_query": None,
        "match_tier": None,
        "error": None,
    }

    if not query_name:
        result.update({
            "match_type": "none", "matched_name": None, "off_id": None,
            "confidence": 0.0, "data_source": "none",
            "match_calories": None, "match_protein": None, "match_fiber": None,
            "match_sugar": None, "match_nova": None,
            "n_candidates_returned": 0, "n_candidates_with_nutrition": 0,
            "n_alternatives_with_nutrition": 0, "alternative_names_with_nutrition": [],
        })
        return result

    try:
        # Tier 1: direct query -- identical call to nutrition_mapping.map_item.
        tier1_candidates = off_api.search_products(query_name)
        tier1_match = matcher.match_product(query_name)

        chosen = None
        tier = None
        pool = tier1_candidates
        query_used = query_name

        if tier1_match is not None:
            chosen = tier1_match
            tier = 1
        else:
            generic = base_terms.generic_term(query_name)
            if generic:
                tier2_candidates = off_api.search_products(generic)
                tier2_match = matcher.match_product(generic, prefer_low_processed=True)
                if tier2_match is not None:
                    # Mirror map_item's relabeling of a generic-term hit.
                    tier2_match.parsed_item_name = query_name
                    tier2_match.match_type = MatchType.FUZZY
                    tier2_match.confidence = round(min(tier2_match.confidence, 0.7), 3)
                    tier2_match.data_source = f"OpenFoodFacts (generic: {generic})"
                    chosen = tier2_match
                    tier = 2
                    pool = tier2_candidates
                    query_used = generic

        if chosen is None:
            chosen = fallback_categories.fallback_nutrition(query_name, category)
            tier = 3

        # Nutrition availability across the candidate pool that decided this outcome.
        pool_with_nutrition = []
        for c in pool:
            nutri = off_api.extract_nutrition(c)
            if matcher._has_usable_nutrition(nutri):
                pool_with_nutrition.append(off_api.product_display_name(c) or "(no name)")

        # "Alternatives" = candidates with nutrition data that were NOT the chosen match.
        alt_names = list(pool_with_nutrition)
        if chosen.matched_name in alt_names:
            alt_names.remove(chosen.matched_name)

        nut = chosen.nutrition
        result.update({
            "off_query": query_used,
            "match_tier": tier,
            "match_type": chosen.match_type.value,
            "matched_name": chosen.matched_name,
            "off_id": chosen.off_id,
            "confidence": chosen.confidence,
            "data_source": chosen.data_source,
            "match_calories": nut.calories_kcal if nut else None,
            "match_protein": nut.protein_g if nut else None,
            "match_fiber": nut.fiber_g if nut else None,
            "match_sugar": nut.sugar_g if nut else None,
            "match_nova": nut.processed_score if nut else None,
            "n_candidates_returned": len(pool),
            "n_candidates_with_nutrition": len(pool_with_nutrition),
            "n_alternatives_with_nutrition": len(alt_names),
            "alternative_names_with_nutrition": alt_names,
        })
    except Exception as e:
        result["error"] = f"{type(e).__name__}: {e}"

    return result

## Run it over every receipt item

Cached OFF lookups make repeat runs fast; a first run against many never-before-seen items will be slower (one OpenFoodFacts request per new query).

In [4]:
rows = []
for i, item in enumerate(items):
    rows.append(investigate_item(item))
    if (i + 1) % 25 == 0 or (i + 1) == len(items):
        print(f"...{i + 1}/{len(items)}")

df = pd.DataFrame(rows)
print(df.shape)

errors = df[df["error"].notna()]
if len(errors):
    print(f"\n{len(errors)} item(s) errored:")
    display(errors[["raw_name", "error"]])

...25/166
...50/166
...75/166
...100/166
...125/166
...150/166
...166/166
(166, 21)


## Full results table

One row per receipt item: raw text -> OFF query -> current match -> alternatives that were passed over.

In [5]:
display_cols = [
    "raw_name", "normalized_name", "off_query", "match_tier", "match_type",
    "matched_name", "off_id", "confidence", "data_source",
    "match_calories", "match_protein", "match_fiber", "match_sugar", "match_nova",
    "n_candidates_returned", "n_candidates_with_nutrition",
    "n_alternatives_with_nutrition", "alternative_names_with_nutrition",
]
df[display_cols]

,raw_name,normalized_name,off_query,match_tier,match_type,matched_name,off_id,confidence,data_source,match_calories,match_protein,match_fiber,match_sugar,match_nova,n_candidates_returned,n_candidates_with_nutrition,n_alternatives_with_nutrition,alternative_names_with_nutrition
0,RUCOLA,Rucola,Rucola,1,fuzzy,Pesto basilic roquette basilicum en rucola,3560071462475,0.850,OpenFoodFacts,428.000000,4.400000,1.800000,2.300000,4.0,5,5,4,"[Pesto Basilico e Rucula, Pizza Rucola Ristorante, Brotaufstrich pflanzlich ..."
1,RADIESCHEN,Radieschen,Radieschen,3,fallback,NaN,NaN,0.300,fallback_category:vegetable,40.000000,2.000000,3.000000,3.000000,1.0,5,0,0,[]
2,LAUCHZWIEBELN,Lauchzwiebeln,Lauchzwiebeln,1,fuzzy,"Couscoussalat mit Paprika, Salzlakenkäse und Lauchzwiebeln",4337256035347,0.850,OpenFoodFacts,210.000000,4.400000,1.800000,3.900000,3.0,5,4,3,"[Obazda Lauchzwiebeln, Bulgursalat mit Lauchzwiebeln und Gemüse, Bulgursalat..."
3,KOHLRABI,Kohlrabi,Kohlrabi,1,fuzzy,Kohlrabi G-Rahm-Gemüse,4250241201186,0.850,OpenFoodFacts,48.000000,1.200000,1.300000,2.100000,3.0,5,3,2,"[Superfood brussels sprouts, napa cabbage, kohlrabi, broccoli, carrots & kal..."
4,BRÜHEN IM GLAS,Brühe im Glas,Brühe im Glas,3,fallback,NaN,NaN,0.300,fallback_category:other,150.000000,3.000000,1.000000,5.000000,3.0,0,0,0,[]
5,KOKOSNUSSMILCH,Kokosnussmilch,Kokosnussmilch,1,exact,Kokosnussmilch,4335619110823,1.000,OpenFoodFacts,170.000000,2.000000,0.000000,2.000000,4.0,5,5,4,"[Kokosnussmilch Klassik, kokosnuss milch, Kokosnussmilch cremig, Kokosnussmi..."
6,FLAMMKUCHENTEIG,Flammkuchenteig,Flammkuchenteig,1,exact,Flammkuchenteig,9120003416217,1.000,OpenFoodFacts,264.000000,7.800000,1.500000,2.700000,4.0,5,5,4,"[Frischer Flammkuchenteig, Flammkuchenteig Dinkel, Flammkuchenteig auf Backp..."
7,PAPRIKA MIX,Paprika Mix,Paprika Mix,1,exact,Chilli & paprika mix,8711299023107,1.000,OpenFoodFacts,484.000000,8.000000,0.000000,7.200000,4.0,5,5,4,[Crevettes purée de patate douce au paprika fumé légumes verts sauce chimich...
8,REISNUDELN 200G,Reisnudeln,Reisnudeln,1,exact,Reisnudeln,8850521950368,1.000,OpenFoodFacts,340.000000,8.000000,2.000000,2.000000,1.0,5,5,4,"[Nudeln-Reisnudeln Pad Thai, Gua Thai Reisnudeln, Nudeln aus Naturreis, Nudeln]"
9,HIRTENKÄSE,Hirtenkäse,Hirtenkäse,1,fuzzy,High Protein Hirtenkäse Jalapeno,4056489451273,0.850,OpenFoodFacts,144.000000,24.500000,NaN,1.300000,NaN,5,5,4,"[High Protein Hirtenkäse, Hirtenkäse traditionell (Gazi), Hirtenkäse Traditi..."


In [9]:
df[df["match_type"]=="fallback"]

,receipt_item_id,receipt_id,raw_name,normalized_name,off_query,match_tier,error,match_type,matched_name,off_id,...,data_source,match_calories,match_protein,match_fiber,match_sugar,match_nova,n_candidates_returned,n_candidates_with_nutrition,n_alternatives_with_nutrition,alternative_names_with_nutrition
1,034facb3-962b-49d2-b812-0e6041bc1f6d,96d33a71-ea37-4d0a-a185-54cd2d1734a0,RADIESCHEN,Radieschen,Radieschen,3,None,fallback,NaN,NaN,...,fallback_category:vegetable,40.0,2.0,3.0,3.0,1.0,5,0,0,[]
4,45925cb6-4357-4dcf-a439-1a522af21a9a,96d33a71-ea37-4d0a-a185-54cd2d1734a0,BRÜHEN IM GLAS,Brühe im Glas,Brühe im Glas,3,None,fallback,NaN,NaN,...,fallback_category:other,150.0,3.0,1.0,5.0,3.0,0,0,0,[]
11,7377b6aa-020a-40e4-9157-73f40e5d236c,2aaff573-a462-4920-8342-8e966f8c2abe,G&G Buttercroiss.,G&G Buttercroissant,G&G Buttercroissant,3,None,fallback,NaN,NaN,...,fallback_category:grain,340.0,9.0,7.0,2.0,2.0,0,0,0,[]
12,7c1ee144-78da-44a7-ad1f-46c50d115665,2aaff573-a462-4920-8342-8e966f8c2abe,G&G Vollk.Sandwich,G&G Vollkorn Sandwich,G&G Vollkorn Sandwich,3,None,fallback,NaN,NaN,...,fallback_category:grain,340.0,9.0,7.0,2.0,2.0,0,0,0,[]
14,d33debe4-5668-4ce8-bb1c-4e233a3a380f,2aaff573-a462-4920-8342-8e966f8c2abe,Tony's Great Bits,Tony's Great Bits,Tony's Great Bits,3,None,fallback,NaN,NaN,...,fallback_category:snack,470.0,6.0,3.0,25.0,4.0,0,0,0,[]
15,c4313223-ca61-4054-bfc4-1fac971e941c,2aaff573-a462-4920-8342-8e966f8c2abe,G&G H-Schmand,G&G H-Schmand,G&G H-Schmand,3,None,fallback,NaN,NaN,...,fallback_category:dairy,90.0,6.0,0.0,5.0,2.0,0,0,0,[]
16,5bfda5b7-bb8b-4a8c-9f5f-e9b245d41c6e,2aaff573-a462-4920-8342-8e966f8c2abe,G&G rei.Butt.,G&G Reibekäse,G&G Reibekäse,3,None,fallback,NaN,NaN,...,fallback_category:dairy,90.0,6.0,0.0,5.0,2.0,0,0,0,[]
17,45998eb7-6031-469f-a5b8-d6c70e484b20,2aaff573-a462-4920-8342-8e966f8c2abe,"AB Bud 6x0,3l MW",AB Bud Bier,AB Bud Bier,3,None,fallback,NaN,NaN,...,fallback_category:drink,42.0,0.0,0.0,9.0,4.0,0,0,0,[]
49,04633f58-e8f9-4c57-9bf0-1afe619bdb65,b7c967e6-6dd5-4963-b767-e4a33d94c897,RADIESCHEN,Radieschen,Radieschen,3,None,fallback,NaN,NaN,...,fallback_category:vegetable,40.0,2.0,3.0,3.0,1.0,5,0,0,[]
51,5abe2ebf-2215-4db0-a32c-c1ef64b65909,b7c967e6-6dd5-4963-b767-e4a33d94c897,CHAMPIGNONS GROSS,Große Champignons,Große Champignons,3,None,fallback,NaN,NaN,...,fallback_category:vegetable,40.0,2.0,3.0,3.0,1.0,0,0,0,[]


## Summary stats

In [ ]:
print("match_type counts:")
print(df["match_type"].value_counts())
print("\nmatch_tier counts (1=direct, 2=generic-term retry, 3=category fallback):")
print(df["match_tier"].value_counts(dropna=False))
print("\naverage # alternatives-with-nutrition per row:", round(df["n_alternatives_with_nutrition"].mean(), 2))
print("rows where at least one unused alternative had nutrition data:",
      (df["n_alternatives_with_nutrition"] > 0).sum(), "/", len(df))

## Diagnostics: likely matching problems

Two views worth checking manually (this is what feeds `MATCH-1`/`MATCH-4` in `backlog_structured.md`):

1. **Fallback despite available OFF data** — the item fell back to a coarse category estimate, but the OFF search it ran *did* return candidates with real nutrition data; they just scored below the similarity threshold. Worth checking whether the threshold or the query text is the problem.
2. **Fuzzy matches with unused alternatives** — a match was accepted, but other candidates with nutrition data existed. Not necessarily wrong, but worth a manual eyeball for cases like "Linsen" -> a wafer snack, where a plainer alternative ("Rote Linsen, getrocknet") was sitting right there.

In [6]:
fallback_missed = df[(df["match_type"] == "fallback") & (df["n_candidates_with_nutrition"] > 0)]
print(f"{len(fallback_missed)} fallback item(s) had usable OFF candidates that scored too low to be chosen:")
fallback_missed[["raw_name", "off_query", "n_candidates_with_nutrition", "alternative_names_with_nutrition"]]

2 fallback item(s) had usable OFF candidates that scored too low to be chosen:


,raw_name,off_query,n_candidates_with_nutrition,alternative_names_with_nutrition
117,"Vollmilch 3,5% 1L",Test Milch,1,[Gouda Jung]
137,"Vollmilch 3,5% 1L","Test Milch 3,5%",1,[Gouda Jung]


In [7]:
fuzzy_with_alts = df[(df["match_type"] != "fallback") & (df["n_alternatives_with_nutrition"] > 0)].sort_values(
    "n_alternatives_with_nutrition", ascending=False
)
print(f"{len(fuzzy_with_alts)} accepted match(es) had at least one unused alternative with nutrition data:")
fuzzy_with_alts[["raw_name", "off_query", "match_type", "matched_name", "confidence",
                 "n_alternatives_with_nutrition", "alternative_names_with_nutrition"]]

137 accepted match(es) had at least one unused alternative with nutrition data:


,raw_name,off_query,match_type,matched_name,confidence,n_alternatives_with_nutrition,alternative_names_with_nutrition
0,RUCOLA,Rucola,fuzzy,Pesto basilic roquette basilicum en rucola,0.850,4,"[Pesto Basilico e Rucula, Pizza Rucola Ristorante, Brotaufstrich pflanzlich ..."
119,Vollkornbrot 500g,Vollkornbrot,exact,Vollkornbrot,1.000,4,"[Pain spécial Vollkornbrot au seigle 500g, Pain noir allemand vollkornbrot 5..."
116,Naturjoghurt 500g,Naturjoghurt,fuzzy,"Joghurt mild 3,5 % Fett",0.850,4,"[Bio Fettarmer Joghurt mild, Cremiger Jogurt - 3,8 % Fett, Joghurt Natur 3,5..."
115,Haferflocken 500g,Haferflocken,exact,Haferflocken,1.000,4,"[Hafer Flocken Zart, Blütenzarte Haferflocken, Zarte Haferflocken, Kernige H..."
114,Gouda jung 200g,Gouda jung,exact,Gouda Jung,1.000,4,"[Gouda jung in Scheiben, Käse Gouda jung, Gouda Jung, Junger Gouda In Scheib..."
113,Bio Eier 10 Stk,Bio Eier,fuzzy,Eier,0.850,4,"[Vegane Salatmayo, Eier, Vegane Mayo, Eier]"
112,"Bananen 1,0 kg",Bananen,fuzzy,Bananenchips,0.850,4,"[Bananen Joghurt mit Knuspermüsli Schoko, Schoko Bananen, Protein Knuspermüs..."
111,Vollkornbrot 500g,Vollkornbrot,exact,Vollkornbrot,1.000,4,"[Pain spécial Vollkornbrot au seigle 500g, Pain noir allemand vollkornbrot 5..."
110,Naturjoghurt 500g,Naturjoghurt,fuzzy,"Joghurt mild 3,5 % Fett",0.850,4,"[Bio Fettarmer Joghurt mild, Cremiger Jogurt - 3,8 % Fett, Joghurt Natur 3,5..."
109,Haferflocken 500g,Haferflocken,exact,Haferflocken,1.000,4,"[Hafer Flocken Zart, Blütenzarte Haferflocken, Zarte Haferflocken, Kernige H..."


## Export (optional)

Writes the full table next to this notebook for offline review / sharing. Not required to run — everything above already displays inline.

In [ ]:
out_path = Path("matching_investigation_results.csv")
df.to_csv(out_path, index=False)
print(f"Saved {len(df)} rows to {out_path.resolve()}")